# 2026 NET_SALES Forecast — Linear Regression by TYPE_OF_DAY (Snowpark input)

**What changed from the month-day version:** instead of matching 2026-09-01 to its *same
calendar date* in 2023/2024/2025 (Sep-01 each year), each date is first mapped to its
**TYPE_OF_DAY** label (e.g. `PP_MINUS_25`, `Pitra_Paksha_3`, `Dussehra`) using the festive-calendar
reference table you provided. The regression is then fit across each series' historical
**TYPE_OF_DAY counterparts** — which usually fall on a *different* calendar date each year, since
festivals shift on the Gregorian calendar year to year.

Example: `2026-09-01` is `PP_MINUS_25`. Its true historical counterparts are **2023-09-04**,
**2024-08-23**, and **2025-08-13** — not `2023-09-01` / `2024-09-01` / `2025-09-01`. This notebook
fits `NET_SALES = slope * YEAR + intercept` across the *TYPE_OF_DAY*-matched points and
extrapolates to 2026.

**Reference table**: embedded directly below (from your `Updated_festive_Dates.xlsx`), so the
notebook is self-contained — no extra file needed at runtime. If you update the festive calendar
later, just re-paste the CSV in the cell marked `REFERENCE_TABLE_CSV`.

**Edge cases handled (same as before):**
- 3 valid years → standard least-squares line
- 2 valid years → line through the 2 points
- 1 valid year → flat-line forecast = that single value
- 0 valid years → left as `NaN` (this happens for a few TYPE_OF_DAY buckets that only exist in
  2026's calendar, e.g. `Post_Dussehra_8`, or whose 2024/2025 counterpart falls before your Sep-01
  data pull started — there's genuinely no history to regress on for those)
- Negative predictions clipped to 0
- Column names normalized to uppercase; `CAL_DATE` parsed flexibly (string or native date/timestamp)

In [ ]:
import numpy as np
import pandas as pd
import io

# If you need to open a Snowpark session yourself in this notebook, uncomment:
# from snowflake.snowpark import Session
# session = Session.builder.configs(connection_parameters).create()
# df = session.table("YOUR_SCHEMA.YOUR_TABLE")

# Otherwise, assume `df` is already a Snowpark DataFrame available in scope,
# e.g. produced by an earlier cell or passed in from your pipeline.
# Required columns: PARENT_DEALER_CODE_MODEL_FAMILY, CAL_DATE, NET_SALES

## 1. Config

In [ ]:
HIST_YEARS   = [2023, 2024, 2025]
PREDICT_YEAR = 2026

## 2. TYPE_OF_DAY reference table (embedded from `Updated_festive_Dates.xlsx`)

Wide format: one row per TYPE_OF_DAY, one column per year's actual calendar date. Blank cells (e.g. `Post_Dussehra_8` for 2023/2024/2025) mean that TYPE_OF_DAY simply didn't occur that year -- handled as missing history, not an error.

In [ ]:
REFERENCE_TABLE_CSV = """TYPE_OF_DAY,CAL_DATE_2023,CAL_DATE_2024,CAL_DATE_2025,CAL_DATE_2026
PP_MINUS_25,2023-09-04,2024-08-23,2025-08-13,2026-09-01
PP_MINUS_24,2023-09-05,2024-08-24,2025-08-14,2026-09-02
PP_MINUS_23,2023-09-06,2024-08-25,2025-08-15,2026-09-03
PP_MINUS_22,2023-09-07,2024-08-26,2025-08-16,2026-09-04
PP_MINUS_21,2023-09-08,2024-08-27,2025-08-17,2026-09-05
PP_MINUS_20,2023-09-09,2024-08-28,2025-08-18,2026-09-06
PP_MINUS_19,2023-09-10,2024-08-29,2025-08-19,2026-09-07
PP_MINUS_18,2023-09-11,2024-08-30,2025-08-20,2026-09-08
PP_MINUS_17,2023-09-12,2024-08-31,2025-08-21,2026-09-09
PP_MINUS_16,2023-09-13,2024-09-01,2025-08-22,2026-09-10
PP_MINUS_15,2023-09-14,2024-09-02,2025-08-23,2026-09-11
PP_MINUS_14,2023-09-15,2024-09-03,2025-08-24,2026-09-12
PP_MINUS_13,2023-09-16,2024-09-04,2025-08-25,2026-09-13
PP_MINUS_12,2023-09-17,2024-09-05,2025-08-26,2026-09-14
PP_MINUS_11,2023-09-18,2024-09-06,2025-08-27,2026-09-15
PP_MINUS_10,2023-09-19,2024-09-07,2025-08-28,2026-09-16
PP_MINUS_9,2023-09-20,2024-09-08,2025-08-29,2026-09-17
PP_MINUS_8,2023-09-21,2024-09-09,2025-08-30,2026-09-18
PP_MINUS_7,2023-09-22,2024-09-10,2025-08-31,2026-09-19
PP_MINUS_6,2023-09-23,2024-09-11,2025-09-01,2026-09-20
PP_MINUS_5,2023-09-24,2024-09-12,2025-09-02,2026-09-21
PP_MINUS_4,2023-09-25,2024-09-13,2025-09-03,2026-09-22
PP_MINUS_3,2023-09-26,2024-09-14,2025-09-04,2026-09-23
PP_MINUS_2,2023-09-27,2024-09-15,2025-09-05,2026-09-24
PP_MINUS_1,2023-09-28,2024-09-16,2025-09-06,2026-09-25
Pitra_Paksha_1,2023-09-29,2024-09-17,2025-09-07,2026-09-26
Pitra_Paksha_2,2023-09-30,2024-09-18,2025-09-08,2026-09-27
Pitra_Paksha_3,2023-10-01,2024-09-19,2025-09-09,2026-09-28
Pitra_Paksha_4,2023-10-02,2024-09-20,2025-09-10,2026-09-29
Pitra_Paksha_5,2023-10-03,2024-09-21,2025-09-11,2026-09-30
Pitra_Paksha_6,2023-10-04,2024-09-22,2025-09-12,2026-10-01
Pitra_Paksha_7,2023-10-05,2024-09-23,2025-09-13,2026-10-02
Pitra_Paksha_8,2023-10-06,2024-09-24,2025-09-14,2026-10-03
Pitra_Paksha_9,2023-10-07,2024-09-25,2025-09-15,2026-10-04
Pitra_Paksha_10,2023-10-08,2024-09-26,2025-09-16,2026-10-05
Pitra_Paksha_11,2023-10-09,2024-09-27,2025-09-17,2026-10-06
Pitra_Paksha_12,2023-10-10,2024-09-28,2025-09-18,2026-10-07
Pitra_Paksha_13,2023-10-11,2024-09-29,2025-09-19,2026-10-08
Pitra_Paksha_14,2023-10-12,2024-09-30,2025-09-20,2026-10-09
Pitra_Paksha_15,2023-10-13,2024-10-01,2025-09-21,2026-10-10
Navratari_1,2023-10-15,2024-10-03,2025-09-22,2026-10-11
Navratari_2,2023-10-16,2024-10-04,2025-09-23,2026-10-12
Navratari_3,2023-10-17,2024-10-05,2025-09-24,2026-10-13
Navratari_4,2023-10-18,2024-10-06,2025-09-25,2026-10-14
Navratari_5,2023-10-19,2024-10-07,2025-09-26,2026-10-15
Navratari_6,2023-10-20,2024-10-08,2025-09-27,2026-10-16
Navratari_7,2023-10-21,2024-10-09,2025-09-28,2026-10-17
Navratari_8,2023-10-22,2024-10-10,2025-09-29,2026-10-18
Navratari_9,2023-10-23,2024-10-11,2025-09-30,2026-10-19
Dussehra,2023-10-24,2024-10-12,2025-10-02,2026-10-20
Post_Dussehra_1,2023-10-25,2024-10-13,2025-10-03,2026-10-21
Post_Dussehra_2,2023-10-26,2024-10-14,2025-10-04,2026-10-22
Post_Dussehra_3,2023-10-27,2024-10-15,2025-10-05,2026-10-23
Post_Dussehra_4,2023-10-28,2024-10-16,2025-10-06,2026-10-24
Post_Dussehra_5,2023-10-29,2024-10-17,2025-10-07,2026-10-25
Post_Dussehra_6,2023-10-30,2024-10-18,2025-10-08,2026-10-26
Post_Dussehra_7,2023-10-31,2024-10-19,2025-10-09,2026-10-27
Post_Dussehra_8,,,,2026-10-28
Karwa_Chauth,2023-11-01,2024-10-20,2025-10-10,2026-10-29
Pre_Dhanteras_1,2023-11-02,2024-10-21,2025-10-11,2026-10-30
Pre_Dhanteras_2,2023-11-03,2024-10-22,2025-10-12,2026-10-31
Pre_Dhanteras_3,2023-11-04,2024-10-23,2025-10-13,2026-11-01
Pre_Dhanteras_4,2023-11-05,2024-10-24,2025-10-14,2026-11-02
Pre_Dhanteras_5,2023-11-06,2024-10-25,2025-10-15,2026-11-03
Pre_Dhanteras_6,2023-11-07,2024-10-26,2025-10-16,2026-11-04
Pre_Dhanteras_7,2023-11-08,2024-10-27,2025-10-17,2026-11-05
Dhanteras,2023-11-10,2024-10-29,2025-10-18,2026-11-06
Pre_Diwali_Day,2023-11-11,2024-10-30,2025-10-19,2026-11-07
Diwali,2023-11-12,2024-10-31,2025-10-20,2026-11-08
Govardhan,2023-11-14,2024-11-02,2025-10-22,2026-11-09
Bhai_dooj,2023-11-15,2024-11-03,2025-10-23,2026-11-10
Post_Bhai_dooj_1,2023-11-16,2024-11-04,2025-10-24,2026-11-11
Post_Bhai_dooj_2,2023-11-17,2024-11-05,2025-10-25,2026-11-12
Post_Bhai_dooj_3,2023-11-18,2024-11-06,2025-10-26,2026-11-13
Post_Bhai_dooj_4,2023-11-19,2024-11-07,2025-10-27,2026-11-14
Post_Bhai_dooj_5,2023-11-20,2024-11-08,2025-10-28,2026-11-15
Post_Bhai_dooj_6,2023-11-21,2024-11-09,2025-10-29,2026-11-16
Post_Bhai_dooj_7,2023-11-22,2024-11-10,2025-10-30,2026-11-17
Post_Bhai_dooj_8,2023-11-23,2024-11-11,2025-10-31,2026-11-18
Post_Bhai_dooj_9,2023-11-24,2024-11-12,2025-11-01,2026-11-19
Post_Bhai_dooj_10,2023-11-25,2024-11-13,2025-11-02,2026-11-20
Post_Bhai_dooj_11,2023-11-26,2024-11-14,2025-11-03,2026-11-21
Post_Bhai_dooj_12,2023-11-27,2024-11-15,2025-11-04,2026-11-22
Post_Bhai_dooj_13,2023-11-28,2024-11-16,2025-11-05,2026-11-23
Post_Bhai_dooj_14,2023-11-29,2024-11-17,2025-11-06,2026-11-24
Post_Bhai_dooj_15,2023-11-30,2024-11-18,2025-11-07,2026-11-25
Post_Bhai_dooj_16,2023-12-01,2024-11-19,2025-11-08,2026-11-26
Post_Bhai_dooj_17,2023-12-02,2024-11-20,2025-11-09,2026-11-27
Post_Bhai_dooj_18,2023-12-03,2024-11-21,2025-11-10,2026-11-28
Post_Bhai_dooj_19,2023-12-04,2024-11-22,2025-11-11,2026-11-29
Post_Bhai_dooj_20,2023-12-05,2024-11-23,2025-11-12,2026-11-30
"""

reference_wide = pd.read_csv(io.StringIO(REFERENCE_TABLE_CSV))
for _c in ["CAL_DATE_2023", "CAL_DATE_2024", "CAL_DATE_2025", "CAL_DATE_2026"]:
    reference_wide[_c] = pd.to_datetime(reference_wide[_c])

reference_wide.head()

## 3. Core functions

In [ ]:
def normalize_columns(pdf: pd.DataFrame) -> pd.DataFrame:
    """Uppercase + strip column names, and confirm the required columns exist."""
    pdf = pdf.copy()
    pdf.columns = [c.strip().upper() for c in pdf.columns]
    required = {"PARENT_DEALER_CODE_MODEL_FAMILY", "CAL_DATE", "NET_SALES"}
    missing = required - set(pdf.columns)
    if missing:
        raise ValueError(f"Missing required columns after normalization: {missing}")
    return pdf


def prep_dates(pdf: pd.DataFrame) -> pd.DataFrame:
    """Parse CAL_DATE (string 'DD-MM-YYYY' OR native date/timestamp) and derive YEAR."""
    pdf = pdf.copy()
    pdf["CAL_DATE_PARSED"] = pd.to_datetime(pdf["CAL_DATE"], dayfirst=True, errors="coerce")

    n_bad = pdf["CAL_DATE_PARSED"].isna().sum()
    if n_bad:
        print(f"[WARN] {n_bad} rows had unparseable CAL_DATE and will be dropped.")
        pdf = pdf.dropna(subset=["CAL_DATE_PARSED"])

    pdf["YEAR"] = pdf["CAL_DATE_PARSED"].dt.year
    pdf["NET_SALES"] = pd.to_numeric(pdf["NET_SALES"], errors="coerce")
    return pdf


def build_reference_lookups(reference_wide: pd.DataFrame, hist_years, predict_year):
    """
    Turn the wide reference table into:
      - ref_long: (TYPE_OF_DAY, YEAR, CAL_DATE) for the historical years -- used to tag each
        historical sales row with its TYPE_OF_DAY.
      - ref_predict: (TYPE_OF_DAY, CAL_DATE_OUT) for the predict year -- used to turn each
        forecasted TYPE_OF_DAY back into a real output date.
    """
    long_df = reference_wide.melt(id_vars="TYPE_OF_DAY", var_name="YEAR_COL", value_name="CAL_DATE")
    long_df["YEAR"] = long_df["YEAR_COL"].str.extract(r"(\d{4})").astype(int)
    long_df = long_df.dropna(subset=["CAL_DATE"])[["TYPE_OF_DAY", "YEAR", "CAL_DATE"]]

    ref_long = long_df[long_df["YEAR"].isin(hist_years)].copy()

    predict_col = f"CAL_DATE_{predict_year}"
    ref_predict = reference_wide[["TYPE_OF_DAY", predict_col]].dropna().rename(
        columns={predict_col: "CAL_DATE_OUT"}
    )
    return ref_long, ref_predict


def tag_with_type_of_day(pdf: pd.DataFrame, ref_long: pd.DataFrame) -> pd.DataFrame:
    """
    Join each historical sales row to its TYPE_OF_DAY via (YEAR, CAL_DATE). Rows whose date
    isn't in the reference table for that year (e.g. before PP_MINUS_25, or after
    Post_Bhai_dooj_20) are dropped -- there's no TYPE_OF_DAY bucket to regress them into.
    """
    merge_key_left = pdf[["PARENT_DEALER_CODE_MODEL_FAMILY", "YEAR", "CAL_DATE_PARSED", "NET_SALES"]].rename(
        columns={"CAL_DATE_PARSED": "CAL_DATE"}
    )
    merged = merge_key_left.merge(ref_long, on=["YEAR", "CAL_DATE"], how="inner")

    n_dropped = len(pdf) - len(merged)
    if n_dropped:
        print(f"[INFO] {n_dropped:,} historical rows fell outside the TYPE_OF_DAY reference "
              f"table for their year and were excluded (e.g. before PP_MINUS_25 or after "
              f"Post_Bhai_dooj_20).")
    return merged


def build_type_of_day_matrix(tagged: pd.DataFrame, hist_years) -> pd.DataFrame:
    """One row per (series, TYPE_OF_DAY), one column per historical year."""
    grouped = (
        tagged.groupby(["PARENT_DEALER_CODE_MODEL_FAMILY", "TYPE_OF_DAY", "YEAR"])["NET_SALES"]
        .mean()
        .reset_index()
    )
    pivot = grouped.pivot_table(
        index=["PARENT_DEALER_CODE_MODEL_FAMILY", "TYPE_OF_DAY"],
        columns="YEAR",
        values="NET_SALES",
    )
    for y in hist_years:
        if y not in pivot.columns:
            pivot[y] = np.nan
    return pivot[hist_years]


def vectorized_linear_forecast(pivot: pd.DataFrame, hist_years, predict_year: int) -> pd.DataFrame:
    """Closed-form OLS across ALL (series, TYPE_OF_DAY) rows at once -- fast even at millions of rows."""
    Y = pivot.to_numpy(dtype=float)
    x = np.array(hist_years, dtype=float)

    mask = ~np.isnan(Y)
    n = mask.sum(axis=1)

    Y_filled = np.where(mask, Y, 0.0)
    X_filled = np.where(mask, np.broadcast_to(x, Y.shape), 0.0)

    sum_x = X_filled.sum(axis=1)
    sum_y = Y_filled.sum(axis=1)
    sum_xy = (X_filled * Y_filled).sum(axis=1)
    sum_x2 = (X_filled * X_filled).sum(axis=1)

    denom = n * sum_x2 - sum_x ** 2

    slope = np.zeros(len(pivot))
    intercept = np.zeros(len(pivot))

    # n >= 2 and not degenerate -> standard closed-form OLS
    valid = (n >= 2) & (denom != 0)
    slope[valid] = (n[valid] * sum_xy[valid] - sum_x[valid] * sum_y[valid]) / denom[valid]
    intercept[valid] = (sum_y[valid] - slope[valid] * sum_x[valid]) / n[valid]

    # exactly 1 point -> flat forecast = that value
    one_point = n == 1
    single_val = np.divide(sum_y, n, out=np.zeros_like(sum_y), where=n > 0)
    intercept[one_point] = single_val[one_point]
    slope[one_point] = 0.0

    # 0 points -> NaN
    no_point = n == 0

    prediction = slope * predict_year + intercept
    prediction[no_point] = np.nan
    prediction = np.where(np.isnan(prediction), prediction, np.clip(prediction, 0, None))

    result = pivot.reset_index()[["PARENT_DEALER_CODE_MODEL_FAMILY", "TYPE_OF_DAY"]].copy()
    result["PREDICTED_NET_SALES"] = prediction
    result["N_HISTORICAL_POINTS"] = n.astype(int)
    return result


def finalize_output(result: pd.DataFrame, ref_predict: pd.DataFrame, predict_year: int) -> pd.DataFrame:
    """Map each forecasted TYPE_OF_DAY back to its real predict-year calendar date."""
    out = result.merge(ref_predict, on="TYPE_OF_DAY", how="inner")  # inner: only TYPE_OF_DAYs that exist in predict_year
    out["CAL_DATE"] = out["CAL_DATE_OUT"].dt.strftime("%d-%m-%Y")

    out = out[
        ["PARENT_DEALER_CODE_MODEL_FAMILY", "TYPE_OF_DAY", "CAL_DATE", "PREDICTED_NET_SALES", "N_HISTORICAL_POINTS"]
    ].sort_values(["PARENT_DEALER_CODE_MODEL_FAMILY", "CAL_DATE"])

    return out.reset_index(drop=True)

## 4. Main pipeline — takes the Snowpark DataFrame `df` in, returns a pandas DataFrame out

In [ ]:
def forecast_from_snowpark_df(snowpark_df, reference_wide=reference_wide,
                               hist_years=HIST_YEARS, predict_year=PREDICT_YEAR) -> pd.DataFrame:
    print("[1/5] Pulling Snowpark DataFrame into pandas ...")
    pdf = snowpark_df.to_pandas()
    pdf = normalize_columns(pdf)
    print(f"      -> {len(pdf):,} rows, {pdf['PARENT_DEALER_CODE_MODEL_FAMILY'].nunique():,} unique series")

    print("[2/5] Parsing dates ...")
    pdf = prep_dates(pdf)

    print("[3/5] Tagging historical rows with TYPE_OF_DAY via the reference calendar ...")
    ref_long, ref_predict = build_reference_lookups(reference_wide, hist_years, predict_year)
    tagged = tag_with_type_of_day(pdf, ref_long)

    print("[4/5] Building (series x TYPE_OF_DAY) x year matrix and fitting regression ...")
    pivot = build_type_of_day_matrix(tagged, hist_years)
    print(f"      -> {len(pivot):,} (series, TYPE_OF_DAY) groups to fit")
    result = vectorized_linear_forecast(pivot, hist_years, predict_year)

    print("[5/5] Mapping forecasts back to real calendar dates ...")
    out = finalize_output(result, ref_predict, predict_year)

    n_no_history = out["PREDICTED_NET_SALES"].isna().sum()
    print(f"      -> Done: {len(out):,} predictions.")
    if n_no_history:
        print(f"      -> NOTE: {n_no_history:,} rows had zero historical data points and are NaN "
              f"(TYPE_OF_DAY buckets with no matching history in 2023-2025).")

    return out

## 5. Run it

In [ ]:
# `df` is your existing Snowpark DataFrame with columns
# PARENT_DEALER_CODE_MODEL_FAMILY, CAL_DATE, NET_SALES
predictions_pdf = forecast_from_snowpark_df(df)
predictions_pdf.head(20)

## 6. (Optional) Write predictions back to Snowflake

If you want the result back as a Snowpark DataFrame / Snowflake table rather than just a local
pandas DataFrame, uncomment and run this (requires an active `session`):

In [ ]:
# predictions_sdf = session.create_dataframe(predictions_pdf)
# predictions_sdf.write.mode("overwrite").save_as_table("YOUR_SCHEMA.NET_SALES_PREDICTIONS_2026_TYPE_OF_DAY")